# 04. Final R = 2 common-eps analysis

This notebook fits or loads the final `R = 2` fixed-eigenmode AR(1)--Binomial model with a common iid residual scale and constructs posterior predictive and posterior-implied copula diagnostics.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import binom, betabinom, norm, rankdata, spearmanr, kendalltau
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(".")
DATA = ROOT / "data"
PDATA = ROOT / "pdata"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
for directory in [DATA, PDATA, FIGURES, TABLES]:
    directory.mkdir(exist_ok=True)

RNG = np.random.default_rng(20260618)
plt.style.use("default")


SECTOR_COLORS = {
    "Basic Materials": "#4C78A8",
    "Capital Goods": "#F58518",
    "Consumer": "#54A24B",
    "Energy": "#E45756",
    "Finance": "#72B7B2",
    "Healthcare": "#B279A2",
    "Technology": "#FF9DA6",
    "Utilities": "#9D755D",
}


def read_sample():
    df = pd.read_csv(DATA / "SAMPLE.csv", parse_dates=["date"])
    expected = ["date", "sector", "obligors", "defaulted"]
    if list(df.columns) != expected:
        raise ValueError(f"Expected columns {expected}, got {list(df.columns)}")
    df = df.sort_values(["date", "sector"]).reset_index(drop=True)
    if (df["obligors"] <= 0).any():
        raise ValueError("obligors must be positive")
    if ((df["defaulted"] < 0) | (df["defaulted"] > df["obligors"])).any():
        raise ValueError("defaulted must lie between zero and obligors")
    return df


def empirical_logit(defaulted, obligors, smooth=0.5):
    defaulted = np.asarray(defaulted, dtype=float)
    obligors = np.asarray(obligors, dtype=float)
    return np.log((defaulted + smooth) / (obligors - defaulted + smooth))


def panel_matrices(df):
    dates = pd.Index(sorted(df["date"].unique()), name="date")
    sectors = pd.Index(sorted(df["sector"].unique()), name="sector")
    dmat = (
        df.pivot(index="date", columns="sector", values="defaulted")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    nmat = (
        df.pivot(index="date", columns="sector", values="obligors")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    x = pd.DataFrame(
        empirical_logit(dmat.to_numpy(), nmat.to_numpy()),
        index=dates,
        columns=sectors,
    )
    rates = dmat / nmat
    return dates, sectors, dmat, nmat, x, rates


def aggregate_k_month(df, k):
    wide_dates = pd.Index(sorted(df["date"].unique()))
    block_lookup = {date: i // k for i, date in enumerate(wide_dates)}
    tmp = df.copy()
    tmp["block"] = tmp["date"].map(block_lookup)
    tmp["block_start"] = tmp["block"].map(lambda b: wide_dates[b * k])
    out = (
        tmp.groupby(["block", "block_start", "sector"], as_index=False)
        .agg(obligors=("obligors", "sum"), defaulted=("defaulted", "sum"))
    )
    out = out.rename(columns={"block_start": "date"})
    out["k_month"] = k
    return out[["date", "sector", "k_month", "obligors", "defaulted"]]


def fit_common_eps_model(df, R=2):
    dates, sectors, dmat, nmat, x, rates = panel_matrices(df)
    X = x.to_numpy()
    intercept = X.mean(axis=0)
    Xc = X - intercept
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    loadings = eigvecs[:, :R]
    scores = Xc @ loadings
    phis = []
    innov_sd = []
    for r in range(R):
        y = scores[1:, r]
        z = scores[:-1, r]
        denom = float(np.dot(z, z))
        phi = float(np.dot(z, y) / denom) if denom > 1e-12 else 0.0
        phi = float(np.clip(phi, -0.98, 0.98))
        resid = y - phi * z
        phis.append(phi)
        innov_sd.append(float(np.sqrt(np.mean(resid**2) + 1e-8)))
    fitted_centered = scores @ loadings.T
    residual = Xc - fitted_centered
    sigma_eps_common = float(np.sqrt(np.mean(residual**2) + 1e-8))
    eta_hat = intercept + fitted_centered
    p_hat = expit(eta_hat)
    ll = float(binom.logpmf(dmat.to_numpy(), nmat.to_numpy(), p_hat).sum())
    n_params = len(sectors) + R * (len(sectors) + 2) + 1
    aic = float(2 * n_params - 2 * ll)
    return {
        "dates": list(dates),
        "sectors": list(sectors),
        "defaulted": dmat,
        "obligors": nmat,
        "logit": x,
        "rates": rates,
        "intercept": intercept,
        "eigvals": eigvals,
        "loadings": loadings,
        "scores": scores,
        "phis": np.array(phis),
        "innov_sd": np.array(innov_sd),
        "sigma_eps_common": sigma_eps_common,
        "eta_hat": eta_hat,
        "p_hat": p_hat,
        "log_likelihood": ll,
        "aic": aic,
    }


def acf_values(x, max_lag=24):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.dot(x, x)
    if denom <= 1e-12:
        return np.zeros(max_lag + 1)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.array(vals)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def beta_binomial_params(rates, obligors, defaulted):
    total_d = float(np.sum(defaulted))
    total_n = float(np.sum(obligors))
    mu = np.clip(total_d / total_n, 1e-6, 1 - 1e-6)
    r = np.asarray(rates, dtype=float)
    v = float(np.nanvar(r, ddof=1)) if len(r) > 1 else 0.0
    base = mu * (1 - mu) / max(float(np.nanmean(obligors)), 1.0)
    extra = max(v - base, 0.0)
    if extra <= 1e-12:
        kappa = 1e5
    else:
        kappa = max(mu * (1 - mu) / extra - 1.0, 2.0)
    return mu * kappa, (1.0 - mu) * kappa


def simulate_dynamic_counts(model, future_obligors, R, n_draws=300, seed=20260618):
    rng = np.random.default_rng(seed)
    sectors = list(model["sectors"])
    H = future_obligors.shape[0]
    S = future_obligors.shape[1]
    last_score = model["scores"][-1, :R].copy()
    loadings = model["loadings"][:, :R]
    draws = np.zeros((n_draws, H, S), dtype=float)
    for m in range(n_draws):
        score = last_score.copy()
        for h in range(H):
            for r in range(R):
                score[r] = model["phis"][r] * score[r] + rng.normal(0.0, model["innov_sd"][r])
            eps = rng.normal(0.0, model["sigma_eps_common"], size=S)
            eta = model["intercept"] + score @ loadings.T + eps
            p = np.clip(expit(eta), 1e-8, 1 - 1e-8)
            draws[m, h, :] = rng.binomial(future_obligors[h, :].astype(int), p)
    return draws

In [ ]:
df = read_sample()
model = fit_common_eps_model(df, R=2)
sectors = list(model["sectors"])

param_rows = []
for s, sector in enumerate(sectors):
    param_rows.append({"parameter": "mu", "sector": sector, "factor": "", "interpretation": "sector intercept", "estimate": model["intercept"][s]})
    for r, label in enumerate(["F1", "F2"]):
        interpretation = "market-wide default-risk factor" if r == 0 else "sector-rotation factor"
        param_rows.append({"parameter": "lambda", "sector": sector, "factor": label, "interpretation": interpretation, "estimate": model["loadings"][s, r]})
for r, label in enumerate(["F1", "F2"]):
    interpretation = "market-wide default-risk factor" if r == 0 else "sector-rotation factor"
    param_rows.append({"parameter": "phi", "sector": "", "factor": label, "interpretation": interpretation, "estimate": model["phis"][r]})
    param_rows.append({"parameter": "innovation_sd", "sector": "", "factor": label, "interpretation": interpretation, "estimate": model["innov_sd"][r]})
param_rows.append({"parameter": "sigma_eps_common", "sector": "", "factor": "", "interpretation": "common iid residual scale", "estimate": model["sigma_eps_common"]})
table4 = pd.DataFrame(param_rows)
table4.to_csv(TABLES / "table5_common_eps_r2_diagnostics.csv", index=False)

serializable = {
    "R": 2,
    "factors": {"F1": "market-wide default-risk factor", "F2": "sector-rotation factor"},
    "sectors": sectors,
    "intercept": model["intercept"].tolist(),
    "loadings": model["loadings"][:, :2].tolist(),
    "phis": model["phis"].tolist(),
    "innov_sd": model["innov_sd"].tolist(),
    "sigma_eps_common": model["sigma_eps_common"],
}
(PDATA / "r2_common_eps_model.json").write_text(json.dumps(serializable, indent=2), encoding="utf-8")
table4.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.6))
scores = model["scores"][:, :2]
ax.plot(scores[:, 0], scores[:, 1], color="#4C78A8", lw=1.0, alpha=0.8)
ax.scatter(scores[0, 0], scores[0, 1], color="#54A24B", s=38, label="Start")
ax.scatter(scores[-1, 0], scores[-1, 1], color="#E45756", s=38, label="End")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("F1: market-wide default-risk factor")
ax.set_ylabel("F2: sector-rotation factor")
ax.set_title("Figure 6. Posterior mean F1--F2 trajectory")
ax.legend(fontsize=8)
savefig(FIGURES / "figure6_posterior_mean_f1_f2_trajectory.png")

In [ ]:
n_draws = 500
future_obligors = model["obligors"].to_numpy()
draw_counts = simulate_dynamic_counts(model, future_obligors, R=2, n_draws=n_draws)
draw_rates = draw_counts / future_obligors[None, :, :]
observed_rates = model["rates"].to_numpy()

pp_mean = draw_rates.mean(axis=0)
pp_low = np.quantile(draw_rates, 0.05, axis=0)
pp_high = np.quantile(draw_rates, 0.95, axis=0)

factor_component = model["scores"][:, :2] @ model["loadings"][:, :2].T
residual_component = model["logit"].to_numpy() - model["intercept"] - factor_component

pp_rows = []
for s, sector in enumerate(sectors):
    obs = observed_rates[:, s]
    pred = pp_mean[:, s]
    factor_var = float(np.var(factor_component[:, s], ddof=1))
    residual_var = float(np.var(residual_component[:, s], ddof=1))
    pp_rows.append({
        "sector": sector,
        "observed_mean_rate": float(np.nanmean(obs)),
        "predictive_mean_rate": float(np.nanmean(pred)),
        "observed_sd_rate": float(np.nanstd(obs, ddof=1)),
        "predictive_sd_rate": float(np.nanmean(np.nanstd(draw_rates[:, :, s], axis=1, ddof=1))),
        "coverage_90": float(np.mean((obs >= pp_low[:, s]) & (obs <= pp_high[:, s]))),
        "factor_variance_share": factor_var / (factor_var + residual_var),
        "common_eps_variance_share": residual_var / (factor_var + residual_var),
    })
table5 = pd.DataFrame(pp_rows)
table5.to_csv(TABLES / "table4_sector_wise_ppc_variance_decomposition.csv", index=False)

pp_summary = []
for t, date in enumerate(model["dates"]):
    for s, sector in enumerate(sectors):
        pp_summary.append({
            "date": pd.Timestamp(date).strftime("%Y-%m-%d"),
            "sector": sector,
            "observed_rate": observed_rates[t, s],
            "predictive_mean_rate": pp_mean[t, s],
            "predictive_q05_rate": pp_low[t, s],
            "predictive_q95_rate": pp_high[t, s],
        })
pd.DataFrame(pp_summary).to_csv(PDATA / "r2_posterior_predictive_summary.csv", index=False)
table5

In [ ]:
dates = pd.to_datetime(model["dates"])
# Store temporal coarse-graining diagnostics without creating an additional manuscript figure.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
obs_all = np.nansum(model["defaulted"].to_numpy(), axis=1) / np.nansum(model["obligors"].to_numpy(), axis=1)
pred_all = np.nansum(pp_mean * model["obligors"].to_numpy(), axis=1) / np.nansum(model["obligors"].to_numpy(), axis=1)
axes[0].plot(dates, obs_all, color="#1F2933", lw=1.1, label="Observed")
axes[0].plot(dates, pred_all, color="#4C78A8", lw=1.1, label="Predictive mean")
axes[0].set_title("Monthly all-sector rate")
axes[0].set_ylabel("Default rate")
axes[0].legend(fontsize=8)

pp_df = pd.DataFrame({"date": dates, "sector": np.repeat("All sectors", len(dates)), "observed_rate": obs_all, "predicted_rate": pred_all})
monthly_all = pd.DataFrame({"date": dates, "sector": "All sectors", "obligors": model["obligors"].sum(axis=1).to_numpy(), "defaulted": model["defaulted"].sum(axis=1).to_numpy()})
coarse_rows = []
for k in [1, 3, 6, 12]:
    obs_k = aggregate_k_month(monthly_all, k)
    obs_k["monthly_equivalent_rate"] = obs_k["defaulted"] / obs_k["obligors"]
    coarse_rows.append({"k_month": k, "observed_var": obs_k["monthly_equivalent_rate"].var(ddof=1)})
pd.DataFrame(coarse_rows).to_csv(PDATA / "r2_temporal_coarse_graining.csv", index=False)
coarse_df = pd.DataFrame(coarse_rows)
axes[1].plot(coarse_df["k_month"], coarse_df["observed_var"], marker="o", color="#F58518")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xticks([1, 3, 6, 12])
axes[1].set_xticklabels(["1", "3", "6", "12"])
axes[1].set_title("Temporal coarse-graining")
axes[1].set_xlabel("Aggregation scale k, months")
axes[1].set_ylabel("All-sector variance")
plt.close(fig)

In [ ]:
rates_df = model["rates"].copy()
pseudo = rates_df.rank(axis=0, method="average") / (len(rates_df) + 1.0)
pseudo.to_csv(PDATA / "copula_pseudo_observations.csv")
z = pd.DataFrame(norm.ppf(pseudo), index=pseudo.index, columns=pseudo.columns)
gaussian_corr = z.corr()

tail_rows = []
for i, a in enumerate(sectors):
    for j, b in enumerate(sectors):
        if j <= i:
            continue
        ua = pseudo[a].to_numpy()
        ub = pseudo[b].to_numpy()
        tail_rows.append({
            "sector_a": a,
            "sector_b": b,
            "spearman": spearmanr(ua, ub).correlation,
            "kendall": kendalltau(ua, ub).correlation,
            "upper_tail_90": float(np.mean((ua > 0.90) & (ub > 0.90)) / 0.10),
            "lower_tail_10": float(np.mean((ua < 0.10) & (ub < 0.10)) / 0.10),
        })
copula_summary = pd.DataFrame(tail_rows)
copula_summary.to_csv(PDATA / "copula_summary.csv", index=False)
gaussian_corr.to_csv(PDATA / "gaussian_copula_correlation.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
a, b = "Finance", "Energy"
axes[0].scatter(pseudo[a], pseudo[b], s=20, alpha=0.65, color="#4C78A8", edgecolor="none")
axes[0].axvline(0.9, color="#E45756", ls="--", lw=1)
axes[0].axhline(0.9, color="#E45756", ls="--", lw=1)
axes[0].set_xlabel(f"Pseudo-observation, {a}")
axes[0].set_ylabel(f"Pseudo-observation, {b}")
axes[0].set_title("Rank-copula pseudo-observations")
x = np.arange(len(table5))
axes[1].bar(x, table5["factor_variance_share"], color="#54A24B", label="Factor share")
axes[1].bar(x, table5["common_eps_variance_share"], bottom=table5["factor_variance_share"], color="#B279A2", label="Common eps share")
axes[1].set_ylim(0, 1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(table5["sector"], rotation=45, ha="right", fontsize=8)
axes[1].set_title("Sector-wise variance decomposition")
axes[1].legend(fontsize=8)
fig.suptitle("Figure 7. Posterior-implied rank copulas")
savefig(FIGURES / "figure7_posterior_implied_rank_copulas.png")

In [ ]:
annual_obs = aggregate_k_month(df, 12)
annual_obs["rate"] = annual_obs["defaulted"] / annual_obs["obligors"]
obs_corr = annual_obs.pivot(index="date", columns="sector", values="rate").corr()

pred_records = []
for t, date in enumerate(model["dates"]):
    for s, sector in enumerate(sectors):
        pred_records.append({
            "date": pd.Timestamp(date),
            "sector": sector,
            "obligors": model["obligors"].iloc[t, s],
            "defaulted": pp_mean[t, s] * model["obligors"].iloc[t, s],
        })
pred_monthly = pd.DataFrame(pred_records)
annual_pred = aggregate_k_month(pred_monthly, 12)
annual_pred["rate"] = annual_pred["defaulted"] / annual_pred["obligors"]
pred_corr = annual_pred.pivot(index="date", columns="sector", values="rate").corr()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, mat, title in [(axes[0], obs_corr, "Observed annual rates"), (axes[1], pred_corr, "Posterior predictive annual rates")]:
    im = ax.imshow(mat, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(np.arange(len(sectors)))
    ax.set_yticks(np.arange(len(sectors)))
    ax.set_xticklabels(sectors, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(sectors, fontsize=8)
    ax.set_title(title)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
fig.suptitle("Figure 5. Annual empirical/PPC correlation matrices")
plt.savefig(FIGURES / "figure5_annual_empirical_ppc_correlation_matrices.png", dpi=180, bbox_inches="tight")
plt.close()